In [1]:
import os
import asyncio
from typing import List, Dict
import dspy
import tiktoken
from dotenv import load_dotenv
import pandas as pd
import openlit
import asyncio
from langfuse import get_client
from openinference.instrumentation.dspy import DSPyInstrumentor
load_dotenv("/home/azureuser/cloudfiles/code/email-generation/.env.local", override=True)

os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_BASE_URL"] = "http://localhost:3000"

langfuse = get_client()
openlit.init(tracer=langfuse._otel_tracer, disable_batch=True)
DSPyInstrumentor().instrument()

In [2]:
api_base = os.getenv("AZURE_LANGUAGE_ENDPOINT")
lm_41_mini = dspy.LM(
    model="azure/gpt-4.1-mini",
    api_key=os.getenv("AZURE_OPENAI_4_1_KEY"),
    api_base=api_base,
    temperature=0.2,
    model_type="responses",
)


In [3]:
class InternalContextCompression(dspy.Signature):
    """
    You are Oddr Context Compressor for law-firm Accounts Receivable notes.
    Compress ONE chunk into 4-5 complete sentences while maitaining important timestamps, invoice numbers, 
    exact dates and other entities such that facts and figures can be referenced and mapped with invoices across the chunks.

    Keep factual details when present:
    - frame this as running timeline by mentioning exact dates and chronology (oldest to newest),
    - invoice numbers and amounts,
    - client/attorney/biller/collector exchanges (who said what),
    - promises, blockers, disputes, escalations, ownership/actions.

    Rules:
    - deduplicate repetition,
    - keep concrete facts,
    - do not fabricate/infer facts,
    - no headings, bullets, or labels.
    """
    internal_context: str = dspy.InputField(desc="One chunk of raw notes/activity text.")
    compressed_notes: str = dspy.OutputField(desc="4-5 sentences factual chunk summary. Always mention the invoice number and date associated with a claim or fact")


class ChunkedClientSummarization(dspy.Signature):
    """
    You are Oddr Summarizer for attorney-facing AR briefings.

    Use only:
    1) financial_data as immutable source of invoice/date/amount facts
    2) notes as joined compressed chunk timeline

    Output constraints:
    - single paragraph
    - exactly 4 complete sentences
    - no headings/bullets/labels
    - date format: Jan 01,2025
    - use selective bolding for high-signal facts

    Sentence order:
    1) AR overview
    2) most recent interaction highlights
    3) minimal historical context only if needed
    4) explicit next-step action with owner/urgency

    If notes are empty, do not imply interactions.
    """
    financial_data: str = dspy.InputField(desc="Immutable financial facts.")
    notes: str = dspy.InputField(desc="Joined compressed chunk notes showing timeline of client interaction, internal attorney notes, and billing "
             "status updates. Use for context reasoning, personalization and ")
    summary: str = dspy.OutputField(desc="Attorney-ready 4-sentence summary paragraph.")


internal_context_compressor = dspy.Predict(InternalContextCompression)
chunk_summarizer = dspy.Predict(ChunkedClientSummarization)


In [4]:
import dspy
from typing import Literal, List

class ClientSummarization(dspy.Signature):
    """
    You are Oddr Summarizer, an AI assistant that helps law firms manage their Accounts Receivable. Your job is to generate clear, concise, and helpful summaries for attorneys using only the facts provided.

    You may infer logical next steps or patterns (e.g., payment gaps, lack of follow-up), but do not fabricate or invent any client interactions or internal updates that aren’t explicitly present in the input.

    If no notes are available, avoid referring to interactions. Focus only on AR status and suggest possible outreach or monitoring based on the financial data.

    Your tone should be professional, factual, and efficient — like a trusted human AR professional writing a briefing. Favor direct, punchy sentences over formal or padded language.

    Given a client with the following financial data and notes across the client, its matters, and invoices, please summarize the client in a way that will make it digestible and meaningful to the time-constrained Attorney responsible for the client so they can understand what is going on with their AR and who is responsible for any next steps.

    Frame this as a timeline, starting with an overview of the client’s current AR status, then highlighting the most recent interaction and any key themes from earlier interactions — but only if such notes are present. Do not fabricate or infer specific updates. However, it’s acceptable to suggest likely follow-up steps or highlight the absence of recent activity if that pattern is evident.

    Reduce repetitive details by grouping similar updates, eliminate specific payment amounts where appropriate, and keep the summary focused and efficient. Mention past events only if they are essential to understanding the current delay or next step. Avoid repeating older context that does not affect present action.

    Format the summary as a single paragraph without any headings or titles, and use smart **bolding** to highlight key points. 
    Limit the summary to **4 complete, efficient sentences**. Favor punchy, high-signal writing over formal explanations. 
    **Do not use headings, bullet points, or labels.** **Mention date always like Jan 01,2025**
    
    Structure the summary as a single flowing paragraph that follows this order:
    [AR overview]
    [Highlights from recent interactions — only if present]
    [Relevant context from earlier interactions — only if helpful]
    [Next steps or follow-up actions]
    """

    financial_data: str = dspy.InputField(
        desc="Structured csv containing Invoice Numbers, Dates, Amounts Outstanding, "
             "and total balance. These figures are immutable facts."
    )

    notes: str = dspy.InputField(
        desc="Mixed timeline of client interaction, internal attorney notes, and billing "
             "status updates. Use for context reasoning, personalization"
    ) 

    summary: str = dspy.OutputField(
        desc="Actionable summary of the Client across the invoices, notes and financial_data. Accurately mention dates, invoice numbers, amounts when summarizing notes and financial_data.  It should be helpful for the attorneys and provides a up-to-date detailed overview of the the account. "
    )


full_summarizer = dspy.Predict(ClientSummarization)

In [5]:
ENCODING = tiktoken.get_encoding("cl100k_base")

def split_by_tokens(text: str, chunk_size_tokens: int = 2000, overlap_tokens: int = 200) -> List[str]:
    text = (text or "").strip()
    if not text:
        return []

    tokens = ENCODING.encode(text)
    if len(tokens) <= chunk_size_tokens:
        return [text]

    step = max(1, chunk_size_tokens - overlap_tokens)
    chunks = []
    for start in range(0, len(tokens), step):
        end = min(start + chunk_size_tokens, len(tokens))
        chunk_text = ENCODING.decode(tokens[start:end]).strip()
        if chunk_text:
            chunks.append(chunk_text)
        if end >= len(tokens):
            break
    return chunks


In [6]:
def compress_notes_to_chunks(notes_text: str, chunk_size_tokens: int = 2000, overlap_tokens: int = 200) -> Dict[str, object]:
    chunks = split_by_tokens(notes_text, chunk_size_tokens=chunk_size_tokens, overlap_tokens=overlap_tokens)
    if not chunks:
        return {"compressed_notes_list": [], "notes_for_summary": ""}

    compressed_notes_list = []
    total = len(chunks)

    for i, chunk in enumerate(chunks, start=1):
        chunk_input = f"Chunk {i}/{total}\\n{chunk}"
        with dspy.context(lm=lm_41_mini.copy(cache=False, temperature=0.2)):
            out = internal_context_compressor(internal_context=chunk_input)
        paragraph = out.compressed_notes.strip()
        if paragraph:
            compressed_notes_list.append(paragraph)

    notes_for_summary = "\\n".join(
        [f"[Chunk {i}/{len(compressed_notes_list)}] {p}" for i, p in enumerate(compressed_notes_list, start=1)]
    )
    return {"compressed_notes_list": compressed_notes_list, "notes_for_summary": notes_for_summary}


async def compress_notes_to_chunks_async(
    notes_text: str,
    chunk_size_tokens: int = 2000,
    overlap_tokens: int = 200,
    max_parallel_calls: int = 5,
) -> Dict[str, object]:
    chunks = split_by_tokens(notes_text, chunk_size_tokens=chunk_size_tokens, overlap_tokens=overlap_tokens)
    if not chunks:
        return {"compressed_notes_list": [], "notes_for_summary": ""}

    total = len(chunks)
    semaphore = asyncio.Semaphore(max_parallel_calls)

    async def _compress_one(i: int, chunk: str):
        chunk_input = f"Chunk {i}/{total}\\n{chunk}"

        def _run():
            with dspy.context(lm=lm_41_mini.copy(cache=False, temperature=0.2)):
                out = internal_context_compressor(internal_context=chunk_input)
            return out.compressed_notes.strip()

        async with semaphore:
            paragraph = await asyncio.to_thread(_run)
        return i, paragraph

    results = await asyncio.gather(
        *[_compress_one(i, chunk) for i, chunk in enumerate(chunks, start=1)]
    )
    results.sort(key=lambda x: x[0])

    compressed_notes_list = [paragraph for _, paragraph in results if paragraph]
    notes_for_summary = "\\n".join(
        [f"[Chunk {i}/{len(compressed_notes_list)}] {p}" for i, p in enumerate(compressed_notes_list, start=1)]
    )

    return {"compressed_notes_list": compressed_notes_list, "notes_for_summary": notes_for_summary}


In [7]:
df = pd.read_csv("/home/azureuser/cloudfiles/code/production-data/stored_completions_5-feb.csv")
df


,completion_id,system_prompt,user_prompt,financial_data,notes,output_summary,notes_token_count
0,chatcmpl-D5OhEPBr9feteTKHFVqVPDGomJmFo,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 1 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,CreatedOn","The client currently has **$20,806.67** in ove...",12
1,chatcmpl-D5Ew1ShgFECUweV0bORKx04cWf8uA,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 3 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...","The client currently has **$16,373.00** in ove...",2168
2,chatcmpl-D5EObN6Z1kzjxrZU2Mx7Wd0OkYq0p,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 2 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...","The client currently has **$6,096.75** in over...",2595
3,chatcmpl-D5ENMYYiOXfT2iEM23SLnJ21at8M1,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 2 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...","The client currently has **$6,096.75** in over...",2595
4,chatcmpl-D5BZKOuyVuimv1c0g4dKPJzdmTNZR,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 3 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...","The client currently has **$16,373.00** in ove...",2169
5,chatcmpl-D3Sv1PJvQJbvPNW72ZUo950OyGHYN,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 2 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...","The client currently has **$13,783.58** in ove...",595
6,chatcmpl-D38mBS1saaIJtVSpHnooddGwj6dH5,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 5 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...",The client currently has **5 active invoices**...,4282
7,chatcmpl-D38JKgRyByi9RVn1f9pKDBNIpyswW,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 15 active invoices, lifetime bi...","ClientId,MatterId,InvoiceNumber,Content,Create...",The client currently has **15 active invoices*...,1266
8,chatcmpl-D370MbDPhgtd2CpQKFiqsDx8M1Ol9,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 16 active invoices, lifetime bi...","ClientId,MatterId,InvoiceNumber,Content,Create...",The client currently has **16 active invoices*...,14972
9,chatcmpl-D363MRXmjwbktBpe12DTb27IIzhlb,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 3 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,CreatedOn","The client currently has **$169,314.63 in over...",12


In [13]:
row_index = 18

financial_data = (df.iloc[row_index]["financial_data"] or "").strip()
notes = (df.iloc[row_index]["notes"] or "").strip()
notes_token_count = (df.iloc[row_index]["notes_token_count"] or "")

print("Notes token count:", notes_token_count)
print("###Financial data:\n", financial_data)
print("\n\n###Notes: \n", notes)

Notes token count: 16404
###Financial data:
 The client has 10 active invoices, lifetime billed amount of $987,957.21, lifetime collected amount of $834,579.11, WIP of $7,090.00, overdue AR of $153,378.10, outstanding AR of $153,378.10, average DSO of 49.


###Notes: 
 ClientId,MatterId,InvoiceNumber,Content,CreatedOn
47833,47833.0001,,"E-mail with no recipient was forwarded to Outlook.\r\nSubject: Outstanding Invoices\r\nMessage: \r\nAttached are copies of invoice numbers <PhoneNumber>, <PhoneNumber>, <PhoneNumber> relating to Client 47833 - <Organization>, Matter 0001 - General Corporate.
-----Imported from Star Collect-----",10/20/2025
47833,47833.0001,,"From: <Email>|Sent: <DateTime>|To: '<Email>'|Cc: <Person>|Subject: RE: May fees - <Organization>||Good afternoon, ||Per your request please see below:||Total amount in Unbilled Fees as of <DateTime> - $43,125.00|Total amount in Unbilled Costs as of <DateTime> - $92.00|Grand Total amount in Unbilled Fees/Costs as of <DateTime> - $43,

In [18]:
chunk_size_tokens = 20000
overlap_tokens = 100
max_parallel_calls = 5

# compressed_notes = compress_notes_to_chunks(
#     notes_text=notes,
#     chunk_size_tokens=chunk_size_tokens,
#     overlap_tokens=overlap_tokens,
# )


compressed_notes = await compress_notes_to_chunks_async(
    notes_text=notes*10,
    chunk_size_tokens=chunk_size_tokens,
    overlap_tokens=overlap_tokens,
    max_parallel_calls=max_parallel_calls,
)

for i, p in enumerate(compressed_notes["compressed_notes_list"], start=1):
    print(f"Chunk {i}: {p}\\n\n\n")

print("\n\nCompressed Notes:\n",compressed_notes["notes_for_summary"])

with dspy.context(lm=lm_41_mini.copy(cache=False, temperature=0.2)):
    chunk_summary = chunk_summarizer(financial_data=financial_data, notes=compressed_notes["notes_for_summary"])
    print(chunk_summary)

Chunk 1: Between September 2024 and October 2025, multiple invoices were issued to Client 47833, Matter 0001 - General Corporate, with notable invoices including #16100097 dated 2/7/2025 for $52,587.40, #16102355 dated 3/12/2025 for $39,225.85, #16104725 dated 4/16/2025 for $42,858.00, and #16105729 dated 5/7/2025 for $39,344.40. As of a 2025 communication, the total outstanding balance was $174,015.65, with two invoices over 60 days past due totaling $91,813.25. The client was repeatedly reminded that invoices are due upon receipt, with payment methods prioritized as ACH, ECHECK, or Wire transfer. There were also internal discussions about adjusting billing procedures, vendor onboarding, and consulting hours tracking for Project Maverick, including instructions for submitting invoices and vendor forms. Additionally, a client name change from Solaris US Bidco to <Organization> was communicated, and requests were made to remove Solaris references from invoices for clarity in payment pro

In [ ]:
with dspy.context(lm=lm_41_mini.copy(cache=False, temperature=0.2)):
    full_summary = full_summarizer(financial_data=financial_data, notes=notes*10)
    print(full_summary)

Prediction(
    summary='The client currently has **10 active invoices with a total outstanding balance of $153,378.10**, and an overdue amount of the same value, reflecting a **lifetime billed amount of $987,957.21 and collected amount of $834,579.11** with an average DSO of 49 days. Recent communications as of **Oct 20, 2025** highlight concern over two invoices totaling $91,813.25 that are over 60 days past due, with a total outstanding balance of $174,015.65 across four invoices dated between **Feb 7, 2025, and May 7, 2025**; payment status is pending review and client has been requested to confirm payment timing. Earlier notes indicate ongoing adjustments in billing procedures and vendor onboarding, including a client name change and invoice submission instructions, with no confirmed payments or disputes noted. Next steps should focus on prompt follow-up for payment on the overdue invoices, confirming receipt and approval of invoices, and ensuring the client’s accounts payable pro

{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "openlit",
                    "telemetry.sdk.version": "1.39.1",
                    "service.name": "default",
                    "deployment.environment": "default"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry.instrumentation.httpx",
                        "version": "0.60b1",
                        "schema_url": "https://opentelemetry.io/schemas/1.11.0",
                        "attributes": null
                    },
                    "metrics": [
                        {
                            "name": "http.client.duration",
                            "description": "measures the duration of the outbound HTTP request",
           